In [1]:
import numpy as np
import pandas as pd
import spacy
from spacy.tokens import DocBin
from tqdm import tqdm
import warnings
warnings.simplefilter("ignore", UserWarning)

nlp = spacy.load("en_core_web_lg")

# ruler = nlp.add_pipe("entity_ruler", after="ner", config={"overwrite_ents": True})
# ruler.from_disk("entity_rulers.jsonl")
# ruler.add_patterns(patterns)


/home/yp6443/anaconda3/envs/sedona/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/yp6443/anaconda3/envs/sedona/lib/python3.10/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/home/yp6443/anaconda3/envs/sedona/lib/python3.10/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


In [2]:
text = ["17:43:26 Delta 276, Tokyo Tower, good evening, taxi to holding point C1."]
for txt in text:
    print(f"text: {txt}")
    doc = nlp(txt)
    for ent in doc.ents:
        print(ent.text, ent.label_)

text: 17:43:26 Delta 276, Tokyo Tower, good evening, taxi to holding point C1.
17:43:26 CARDINAL
Delta 276 ORG
Tokyo Tower FAC
evening TIME
point C1 PRODUCT


In [3]:
# ! python -m spacy init config config.cfg --lang en --pipeline ner --optimize efficiency --force

In [4]:
! python -m spacy train config.cfg --output ./tok2vec --paths.train ./train_data.spacy --paths.dev ./val_data.spacy

/home/yp6443/anaconda3/envs/sedona/lib/python3.10/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
ℹ Saving to output directory: tok2vec
ℹ Using CPU
ℹ To switch to GPU 0, use the option: --gpu-id 0

=========================== Initializing pipeline ===========================
/home/yp6443/anaconda3/envs/sedona/lib/python3.10/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
✔ Initialized pipeline

============================= Training pipeline =============================
ℹ Pipeline: ['tok2vec', 'ner']
ℹ Initial learn rate: 0.001
E    #       LOSS TOK2VEC  LOSS NER  ENTS_F  ENTS_P  ENTS_R  SCORE 
---  ------  ------------  --------  ------

In [5]:
from spacy.training import Corpus

nlp_ner = spacy.load("./tok2vec/model-best")
print(nlp_ner.pipe_names)

# Load test data (adjust path to your test data)
test_data_path = "/home/yp6443/research/nlp/taxigen/test_data.spacy"

# If your test data is already in .spacy format:0.8
corpus = Corpus(test_data_path)
test_examples = corpus(nlp_ner)

# Evaluate the model
scores = nlp_ner.evaluate(test_examples)

# Print NER metrics
print("=== Tok2Vec w/o Heuristics ===")
print("=== NER Evaluation Metrics ===")
print(f"Precision: {scores['ents_p']:.3f}")    # Use ['ents_p']
print(f"Recall:    {scores['ents_r']:.3f}")    # Use ['ents_r']
print(f"F1-Score:  {scores['ents_f']:.3f}")    # Use ['ents_f']

['tok2vec', 'ner']
=== Tok2Vec w/o Heuristics ===
=== NER Evaluation Metrics ===
Precision: 0.859
Recall:    0.559
F1-Score:  0.677


In [6]:
from memory_profiler import memory_usage
import time

def test_inference(text: str):
    nlp_ner = spacy.load("./tok2vec/model-best")
    
    start_time = time.time()
    doc = nlp_ner(text)
    print(f"Time taken: {time.time() - start_time:.2f} seconds")
    # Return something if needed
    return doc

# 1) Measure memory before/after running the function
test_text = "Japan Air 179, Tokyo Tower, good evening, number 3, taxi to holding point C1."

start_mem = memory_usage()[0]  # baseline memory
doc = test_inference(test_text)
end_mem = memory_usage()[0]    # memory after running

print(f"Memory used: {end_mem - start_mem:.2f} MiB\n")

print("Entities found:")
for ent in doc.ents:
    print(f" - {ent.text} : {ent.label_}")

Time taken: 0.03 seconds
Memory used: 56.94 MiB

Entities found:
 - Japan Air 179 : CALLSIGN
 - taxi : ACSTATE
 - holding : ACSTATE


In [7]:
import glob
test_file = glob.glob('/home/yp6443/research/nlp/voice_data/test_file/*.txt')
file_idx = 1

with open(test_file[file_idx], 'r') as file:
    for line in file:
        doc = nlp_ner(line.strip())
        spacy.displacy.render(doc, style="ent", jupyter=True)

In [8]:
nlp_ner = spacy.load("./tok2vec/model-best")

ruler = nlp_ner.add_pipe(
    "entity_ruler",
    after="ner",  # or before="ner"
    config={"overwrite_ents": True}  
)
ruler.from_disk("entity_rulers.jsonl")

with open(test_file[file_idx], 'r') as file:
    for line in file:
        doc = nlp_ner(line.strip())
        spacy.displacy.render(doc, style="ent", jupyter=True)

In [9]:
nlp_ner = spacy.load("./tok2vec/model-best")

ruler = nlp_ner.add_pipe(
    "entity_ruler",
    after="ner",  # or before="ner"
    config={"overwrite_ents": False}  
)
ruler.from_disk("entity_rulers.jsonl")
print(nlp_ner.pipe_names)

# Load test data (adjust path to your test data)
test_data_path = "/home/yp6443/research/nlp/taxigen/test_data.spacy"

# If your test data is already in .spacy format:
corpus = Corpus(test_data_path)
test_examples = corpus(nlp_ner)

# Evaluate the model
scores = nlp_ner.evaluate(test_examples)

# Print NER metrics
print("=== Tok2Vec w/ Heuristics ===")
print("=== NER Evaluation Metrics ===")
print(f"Precision: {scores['ents_p']:.3f}")    
print(f"Recall:    {scores['ents_r']:.3f}")
print(f"F1-Score:  {scores['ents_f']:.3f}")    


['tok2vec', 'ner', 'entity_ruler']
=== Tok2Vec w/ Heuristics ===
=== NER Evaluation Metrics ===
Precision: 0.805
Recall:    0.678
F1-Score:  0.736


In [10]:
nlp_ner = spacy.load("./tok2vec/model-best")

ruler = nlp_ner.add_pipe(
    "entity_ruler",
    after="ner",  # or before="ner"
    config={"overwrite_ents": False}  
)
ruler.from_disk("entity_rulers.jsonl")
print(nlp_ner.pipe_names)

# Load test data (adjust path to your test data)
test_data_path = "/home/yp6443/research/nlp/taxigen/test_data.spacy"

# If your test data is already in .spacy format:
corpus = Corpus(test_data_path)
test_examples = corpus(nlp_ner)

# Evaluate the model
scores = nlp_ner.evaluate(test_examples)

# Print NER metrics
print("=== Tok2Vec w/ Heuristics ===")
print("=== NER Evaluation Metrics ===")
print(f"Precision: {scores['ents_p']:.3f}")    
print(f"Recall:    {scores['ents_r']:.3f}")
print(f"F1-Score:  {scores['ents_f']:.3f}")    

['tok2vec', 'ner', 'entity_ruler']
=== Tok2Vec w/ Heuristics ===
=== NER Evaluation Metrics ===
Precision: 0.805
Recall:    0.678
F1-Score:  0.736


In [11]:
from spacy.training import Example
from spacy import load

corpus = Corpus("/home/yp6443/research/nlp/taxigen/test_data.spacy")

for i, eg in enumerate(corpus(nlp_ner), start=1):
    print(f"Example #{i}")
    print("Text:", eg.reference.text)
    print("Gold entities:", eg.reference.ents)
    if i >= 5:
        break

Example #1
Text: 0:08 And Delta 295 heavy taxi with Romeo.
Gold entities: (Delta 295, taxi, Romeo)
Example #2
Text: 0:14 the 295 heavy Atlanta ground run my 8 ride taxi golf short of foxtrot.
Gold entities: (295, taxi, foxtrot)
Example #3
Text: 0:20 Taxi via golf certified or 295.
Gold entities: (Taxi, golf)
Example #4
Text: 0:33 295 heavy turn on foxtrot hold short of ramp 5.
Gold entities: (295, turn, foxtrot, hold, ramp 5)
Example #5
Text: 0:36 Alright, make a left on Foxtrot short of ramp 5 Delta 295 
Gold entities: (left, Foxtrot, ramp 5, Delta 295)


# Training with Entity Rule Spans in the training dataset

In [12]:
! python -m spacy info

/home/yp6443/anaconda3/envs/sedona/lib/python3.10/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(

============================== Info about spaCy ==============================

spaCy version    3.8.4                         
Location         /home/yp6443/anaconda3/envs/sedona/lib/python3.10/site-packages/spacy
Platform         Linux-6.5.0-41-generic-x86_64-with-glibc2.35
Python version   3.10.16                       
Pipelines        en_core_web_lg (3.8.0), en_core_web_sm (3.8.0), en_core_web_trf (3.8.0)



In [13]:
# ! python -m spacy init config config_tok_rule.cfg \
#   --lang en \
#   --pipeline entity_ruler,ner \
#   --optimize efficiency \
#   --force

In [14]:
! python -m spacy train config.cfg --output ./tok2vec_rules --paths.train ./augmented_train_data.spacy --paths.dev ./val_data.spacy

/home/yp6443/anaconda3/envs/sedona/lib/python3.10/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
ℹ Saving to output directory: tok2vec_rules
ℹ Using CPU
ℹ To switch to GPU 0, use the option: --gpu-id 0

=========================== Initializing pipeline ===========================
/home/yp6443/anaconda3/envs/sedona/lib/python3.10/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
✔ Initialized pipeline

============================= Training pipeline =============================
ℹ Pipeline: ['tok2vec', 'ner']
ℹ Initial learn rate: 0.001
E    #       LOSS TOK2VEC  LOSS NER  ENTS_F  ENTS_P  ENTS_R  SCORE 
---  ------  ------------  --------  

In [15]:
from spacy.training import Corpus

nlp_ner = spacy.load("./tok2vec_rules/model-best")

# Load test data (adjust path to your test data)
test_data_path = "/home/yp6443/research/nlp/taxigen/test_data.spacy"

# If your test data is already in .spacy format:
corpus = Corpus(test_data_path)
test_examples = corpus(nlp_ner)

# Evaluate the model
scores = nlp_ner.evaluate(test_examples)

# Print NER metrics
print("=== Tok2Vec trained w/ Heuristics w/o Heuristics override ===")
print("=== NER Evaluation Metrics ===")
print(f"Precision: {scores['ents_p']:.3f}")    
print(f"Recall:    {scores['ents_r']:.3f}")
print(f"F1-Score:  {scores['ents_f']:.3f}")    

=== Tok2Vec trained w/ Heuristics w/o Heuristics override ===
=== NER Evaluation Metrics ===
Precision: 0.882
Recall:    0.638
F1-Score:  0.740


In [16]:
from memory_profiler import memory_usage
import time

def test_inference(text: str):
    nlp_ner = spacy.load("./tok2vec_rules/model-best")
    start_time = time.time()
    doc = nlp_ner(text)
    print(f"Time taken: {time.time() - start_time:.2f} seconds")
    # Return something if needed
    return doc

# 1) Measure memory before/after running the function
test_text = "Japan Air 179, Tokyo Tower, good evening, number 3, taxi to holding point C1."

start_mem = memory_usage()[0]  # baseline memory
doc = test_inference(test_text)
end_mem = memory_usage()[0]    # memory after running

print(f"Memory used: {end_mem - start_mem:.2f} MiB\n")

print("Entities found:")
for ent in doc.ents:
    print(f" - {ent.text} : {ent.label_}")

Time taken: 0.02 seconds
Memory used: 55.98 MiB

Entities found:
 - Japan Air 179 : CALLSIGN
 - taxi : ACSTATE
 - holding : ACSTATE


In [17]:
nlp_ner = spacy.load("./tok2vec_rules/model-best")

ruler = nlp_ner.add_pipe(
    "entity_ruler",
    after="ner",  # or before="ner"
    config={"overwrite_ents": False}  
)
ruler.from_disk("entity_rulers.jsonl")
print(nlp_ner.pipe_names)

# Load test data (adjust path to your test data)
test_data_path = "/home/yp6443/research/nlp/taxigen/test_data.spacy"

# If your test data is already in .spacy format:
corpus = Corpus(test_data_path)
test_examples = corpus(nlp_ner)

# Evaluate the model
scores = nlp_ner.evaluate(test_examples)

# Print NER metrics
print("=== Tok2Vec trained w/ Heuristics w/ Heuristics override ===")
print("=== NER Evaluation Metrics ===")
print(f"Precision: {scores['ents_p']:.3f}")    
print(f"Recall:    {scores['ents_r']:.3f}")
print(f"F1-Score:  {scores['ents_f']:.3f}")    


['tok2vec', 'ner', 'entity_ruler']
=== Tok2Vec trained w/ Heuristics w/ Heuristics override ===
=== NER Evaluation Metrics ===
Precision: 0.878
Recall:    0.757
F1-Score:  0.813


In [18]:
import spacy
from memory_profiler import memory_usage
import time

def test_inference(text: str):
    nlp_ner = spacy.load("./tok2vec_rules/model-best")
    ruler = nlp_ner.add_pipe(
    "entity_ruler",
    after="ner",  # or before="ner"
    config={"overwrite_ents": False}  
    )
    ruler.from_disk("entity_rulers.jsonl")

    start_time = time.time()
    doc = nlp_ner(text)
    print(f"Time taken: {time.time() - start_time:.2f} seconds")
    # Return something if needed
    return doc

# 1) Measure memory before/after running the function
test_text = "Japan Air 179, Tokyo Tower, good evening, number 3, taxi to holding point C1."

start_mem = memory_usage()[0]  # baseline memory
doc = test_inference(test_text)
end_mem = memory_usage()[0]    # memory after running

print(f"Memory used: {end_mem - start_mem:.2f} MiB\n")

print("Entities found:")
for ent in doc.ents:
    print(f" - {ent.text} : {ent.label_}")

Time taken: 0.02 seconds
Memory used: 0.00 MiB

Entities found:
 - Japan Air 179 : CALLSIGN
 - taxi : ACSTATE
 - holding : ACSTATE
